## Offline constructed route

This Notebook runs the constructed three-pair main pipeline and saves one new spread figure. It does not select a real-data branch. Real-data files are not bundled, and the rights and permitted uses have not been established.

[Entry modes and preparation](../../../NOTEBOOK_GUIDE.md). See this guide for run steps, data requirements, and expected outputs.

# Project 5: walk-forward pairs and an execution ledger

Forward-built total returns are distinct from raw execution prices. Rolling formation uses 504 observations, with 126-observation trading windows and frozen parameters. Entry/exit/stop spread thresholds are ±2, ±0.5 and ±4. The v2 selection rule is EG p<0.05 AND beta>0, defining long/short relative value. It was frozen before real data but after constructed results were seen, so it is not a pristine preregistration of the constructed experiment.

The ledger distinguishes signal and fill clocks, matched/independent partial fills, open marks, ACT/360 financing, borrow and dividend cash. A 5%-of-known-ADV20 opening budget is an uncalibrated ex-ante scenario, not guaranteed realized participation. Reconciliation checks do not prove zero leakage in every possible input.

In [ ]:
from pathlib import Path
import os, sys, uuid
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
for path in (ROOT / "src", ROOT / "code", ROOT):
    if str(path) not in sys.path: sys.path.insert(0, str(path))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("p05_pairs_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Run directory:", RUN_DIR.relative_to(ROOT))
import json, hashlib, numpy as np, pandas as pd
import matplotlib.pyplot as plt
root = ROOT / "projects/project05"
for path in (root, root / "src"): sys.path.insert(0,str(path))
import p05_constructed
import p05_pairs as pp
import run_project05


## 1. Configuration and constructed inputs

Read preregistration_v2.json and its byte hash. The code explicitly calls load_inputs('constructed', prereg) for three pairs. use_real=False is not a permission switch and a frozen real manifest would not itself authorize execution.

In [ ]:
prereg_path = root / 'configs' / 'preregistration_v2.json'
prereg_bytes = prereg_path.read_bytes()
prereg = json.loads(prereg_bytes)
prereg_digest = hashlib.sha256(prereg_bytes).hexdigest()
manifest_path = root / 'data_manifest.json'

# Always choose the offline constructed branch; real-data execution needs separate permission.
use_real = False
panel_df, rates_df, pairs, meta = run_project05.load_inputs('constructed', prereg)
print(f'CONSTRUCTED teaching panel ({meta["data_class"]}): {len(pairs)} pairs; real route SKIPPED')


## 2. Fresh main simulation

Run all three constructed pairs through rolling formation/trading windows. Compare screened and ungated sleeves. Screening uses p<0.05 together with positive beta; the BH q-value is reported separately and does not silently replace the selected rule. This run is not the complete cost/threshold sensitivity grid.

In [ ]:
results = run_project05.run_all(panel_df, rates_df, pairs, prereg)
print(f'Walk-forward completed: {len(results["windows"])} rolling windows.')
print(f'Total formation tests: {len(results["formations"])}')
print(f'EG p < 0.05: {int(results["formations"]["eg_pass"].sum())}; selected (p < 0.05 and beta > 0): {int(results["formations"]["selected"].sum())}')

## 3. Formation table

Read sample size, intercept, hedge ratio, spread SD, half-life, level correlation, EG statistic/p-value, within-window BH q-value and separate EG-pass/selected flags. A significant relation is not a profitability test.

In [ ]:
df_form = results['formations'][[
    'pair', 'window', 'n', 'alpha', 'beta', 'spread_sd', 'half_life_days',
    'corr_level', 'eg_stat', 'eg_pvalue', 'qvalue_bh_window', 'eg_pass', 'selected'
]].copy()
print(df_form.to_string(index=False))

## 4. Spread and execution markers: two clocks

The line is z on signal dates for pair CA_CB in representative window 2. Each marker is placed horizontally at fill_date, but its vertical z comes from the preceding signal_date. It connects a delayed fill to its originating signal; it is not z observed on the fill date. The shared plotting code is preserved. Save pairs_spread_CONSTRUCTED.png under RUN_DIR.

In [ ]:
target_pair = 'CA_CB'
w_target = results['windows'][2]  # Representative window 2
form_target = results['formations'][(results['formations']['pair'] == target_pair) & (results['formations']['window'] == w_target.k)].iloc[0]

P = results['P']
log_tr = np.log(pp.total_return_index(P))
sig_dates = w_target.signal_dates
s = log_tr.loc[sig_dates, form_target['y']] - form_target['alpha'] - form_target['beta'] * log_tr.loc[sig_dates, form_target['x']]
z = (s - form_target['spread_mean']) / form_target['spread_sd']

fig, ax = plt.subplots(figsize=(10, 4.5), dpi=150)
ax.plot(z.index, z.values, color='steelblue', linewidth=1.8, label='Spread z-score')
ax.axhline(2.0, color='crimson', linestyle='--', label='Entry (+2.0)')
ax.axhline(0.5, color='darkorange', linestyle=':', label='Exit (+0.5)')
ax.axhline(0.0, color='gray', linestyle='-', linewidth=0.8)
ax.axhline(-0.5, color='darkorange', linestyle=':', label='Exit (-0.5)')
ax.axhline(-2.0, color='crimson', linestyle='--', label='Entry (-2.0)')

orders = results['orders']
sub_o = orders[(orders['pair'] == target_pair) & (orders['gate'] == 'screened')]
if len(sub_o):
    for _, row_o in sub_o.iterrows():
        sd = pd.Timestamp(row_o['signal_date'])
        if sd in z.index:
            zv = z.loc[sd]
            color = 'green' if row_o['kind'] == 'entry' else 'purple'
            marker = '^' if (row_o['kind'] == 'entry' and zv < 0) else ('v' if row_o['kind'] == 'entry' else 'o')
            ax.scatter(pd.Timestamp(row_o['fill_date']), zv, color=color, marker=marker, s=70, zorder=5)

ax.set_title(f'Pair {target_pair} Spread Dynamics & Ledger Execution (Window {w_target.k})\n[CONSTRUCTED — not market evidence]', fontsize=11, fontweight='bold')
ax.set_xlabel('Trading Date (YYYY-MM)')
ax.set_ylabel('Spread z-score (SD)')
ax.legend(loc='upper right', fontsize=8)
ax.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
fig.savefig(RUN_DIR / "pairs_spread_CONSTRUCTED.png", dpi=130)
plt.show()

## 5. Cash reconciliation

Check cash minus (initial wealth + interest − borrow + dividends − trade flow − commission) within tolerance. Spread and impact are embedded in fill prices; they are displayed for attribution, not deducted a second time.

In [ ]:
reconciliation = []
for (pair, gate), led in results['ledgers'].items():
    reconciliation.append({
        'pair': pair,
        'gate': gate,
        'interest': led['totals']['interest'],
        'borrow': led['totals']['borrow'],
        'dividends': led['totals']['dividends'],
        'commission': led['totals']['commission'],
        'spread_cost': led['totals']['spread_cost'],
        'impact_cost': led['totals']['impact_cost'],
        'cash_residual': led['cash_residual']
    })
df_rec = pd.DataFrame(reconciliation)
print('LEDGER RECONCILIATION SUMMARY:')
print(df_rec[['pair', 'gate', 'interest', 'borrow', 'dividends', 'commission', 'cash_residual']].to_string(index=False))
assert (df_rec['cash_residual'].abs() < 1e-7).all(), 'Non-zero cash residual detected!'

## 6. Constructed metrics

Read total/annual return, volatility, excess Sharpe over daily cash credit, drawdown, open/closed trades, holding days, declared turnover convention and cash residual. These are outputs of the constructed main route, not market observations or authorization for a real route.

In [ ]:
metrics_df = run_project05.metrics_table(results)
print('HEADLINE METRICS TABLE [CONSTRUCTED — NOT MARKET EVIDENCE]:')
cols = ['pair', 'gate', 'n_returns', 'total_return', 'ann_return', 'ann_vol', 'sharpe_excess', 'max_drawdown', 'closed_trades', 'open_trades', 'win_rate', 'mean_holding_days', 'turnover_one_sided_ann', 'cash_residual']
print(metrics_df[cols].to_string(index=False))